In [ ]:
# (필수) 노트북 위치와 무관하게 src 패키지를 import 할 수 있게 경로를 잡습니다.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project ROOT =", ROOT)


# 09 — Model Extraction (Toy)

예측 API 기반 모델 추출 원리(선형 모델) 데모.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

rng = np.random.RandomState(0)
X = rng.normal(size=(5000, 10))
true_w = rng.normal(size=(10,))
y = (X @ true_w + 0.2*rng.normal(size=(5000,)) > 0).astype(int)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0)
teacher = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("teacher acc:", accuracy_score(yte, teacher.predict(Xte)))

def predict_api_proba(x_batch):
    return teacher.predict_proba(x_batch)[:, 1]

def predict_api_rounded(x_batch, decimals=1):
    return np.round(predict_api_proba(x_batch), decimals=decimals)

def predict_api_label_only(x_batch):
    return (predict_api_proba(x_batch) > 0.5).astype(int)

def steal(api_fn, n_queries=3000, mode="proba"):
    Xq = rng.normal(size=(n_queries, 10))
    yq = api_fn(Xq)
    if mode == "proba":
        yq = (yq > 0.5).astype(int)
    return Xq, yq

def train_surrogate(Xq, yq):
    return LogisticRegression(max_iter=1000).fit(Xq, yq)

def eval_surrogate(sur):
    return accuracy_score(yte, sur.predict(Xte))

Xq, yq = steal(predict_api_proba, mode="proba")
sur = train_surrogate(Xq, yq)
print("surrogate acc (proba):", eval_surrogate(sur))

Xq2, yq2 = steal(lambda xb: predict_api_rounded(xb, decimals=1), mode="proba")
sur2 = train_surrogate(Xq2, yq2)
print("surrogate acc (rounded):", eval_surrogate(sur2))

Xq3, yq3 = steal(predict_api_label_only, mode="label")
sur3 = train_surrogate(Xq3, yq3)
print("surrogate acc (label-only):", eval_surrogate(sur3))
